# Topic Visualization

Same UMAP projection as `visualize_embeddings.ipynb`, coloured by BERTopic topic.

In [1]:
import mailbox
import numpy as np
from pathlib import Path

from bertopic import BERTopic
from usenet_no.mbox_utils import message_factory, get_message_body

MODEL = "codefuse-ai/F2LLM-v2-0.6B"
RUN_TAG = "min400_max600_nr30"  # matches topic_modelling.py --min-messages 400 --max-messages 600

embedding_dir = Path(f"../data/embeddings/{MODEL}")
topics_dir = Path(f"../data/topics/{MODEL}/{RUN_TAG}")
source_dirs = {
    "ia": Path("../data/internet_archive/utf_8_data"),
    "nwa": Path("../data/nwa_90s/utf_8_data"),
}

all_embeddings = []
embedding_indexer = []
text_indexer = []

for f in sorted(embedding_dir.iterdir()):
    if f.stem.endswith("_index"):
        continue
    embs = np.load(f)
    if len(embs) < 400 or len(embs) > 600:
        continue

    mbox_stem, source = f.stem.rsplit("_", 1)
    mbox_file = source_dirs[source] / f"{mbox_stem}.mbox"
    messages = list(mailbox.mbox(str(mbox_file), factory=message_factory))

    index_file = embedding_dir / f"{f.stem}_index.npy"
    if index_file.exists():
        indices = np.load(index_file)
        bodies = [get_message_body(messages[i]) for i in indices]
    else:
        bodies = [get_message_body(m) for m in messages]

    all_embeddings.extend(embs)
    embedding_indexer += [f.stem] * len(embs)
    text_indexer += bodies

all_embeddings = np.array(all_embeddings)
print(f"Loaded {len(text_indexer)} documents, embeddings shape: {all_embeddings.shape}")

Loaded 10066 documents, embeddings shape: (10066, 1024)


In [2]:
umap_2d = np.load(embedding_dir / "umap_2d_visualization.npy")

topic_model = BERTopic.load(str(topics_dir / "bertopic_model"))
topics, _ = topic_model.transform(text_indexer, all_embeddings)
topics = np.array(topics)

topic_info = topic_model.get_topic_info().set_index("Topic")

print(f"UMAP shape: {umap_2d.shape}")
print("Unique topics:")
for t in sorted(set(topics)):
    if t == -1:
        print(f"  -1: outliers: {topic_info.loc[t, 'Representation'][:5]}")
    else:
        words = ", ".join(topic_info.loc[t, "Representation"][:5])
        print(f"  {t}: {words}")

UMAP shape: (10066, 2)
Unique topics:
  -1: outliers: ['det', 'er', 'og', 'en', 'som']
  0: det, er, og, som, at
  1: er, det, jeg, som, og
  2: stk, en, selges, sun, med
  3: en, det, er, har, og
  4: the, you, to, and, your
  5: 1c, of, gold, order, 708108q5t48epa
  6: celebrity, skinny, man, the, free
  7: nye, gruppen, url, noittjenestermaildiverse, nofolkloreovertro
  8: ambrina, herbal, erectile, dysfunction, male
  9: torrentti, gratis, terjejmailandnewscom, freeyellow, jonassen
  10: bygdetun, ronald, ordet, museum, eller
  11: uuencodedportionremoved, blah, extortion, cx7u0828n50, testzip
  12: mbone, multimedia, multicast, kan, av
  13: fiske, akvarium, fluefiske, fisker, har
  14: test, blah, bye, bla, remove
  15: java, turbo, pascal, tjelta, kent
  16: the, was, and, military, to
  17: httms, makevariablelocal, setq, makevariablebufferlocal, timestamp
  18: descrambler, cable, the, shack, to
  19: taste, vaere, hrefhttpwwwoslonettno, aa, la
  20: amazonene, jungler, kvinnf

In [3]:
import colorsys
import plotly.graph_objects as go


def hsl_to_hex(h, s, lightness):
    r, g, b = colorsys.hls_to_rgb(h / 360, lightness / 100, s / 100)
    return f"#{int(r * 255):02x}{int(g * 255):02x}{int(b * 255):02x}"


def topic_label(t, n_words=5):
    if t == -1:
        return "outliers (-1)"
    words = ", ".join(topic_info.loc[t, "Representation"][:n_words])
    return f"Topic {t}: {words}"


topic_info = topic_model.get_topic_info().set_index("Topic")
unique_topics = sorted(set(topics))
color_map = {
    t: "lightgrey"
    if t == -1
    else hsl_to_hex(int(i * 360 / (len(unique_topics) - 1)), 70, 50)
    for i, t in enumerate(unique_topics)
}

hover_texts = np.array(
    [
        f"<b>{topic_label(t)}</b><br><i>{stem.rsplit('_', 1)[0]}</i><br>"
        + body[:400].replace("\n", "<br>")
        for t, stem, body in zip(topics, embedding_indexer, text_indexer)
    ]
)

fig = go.Figure()

for t in unique_topics:
    mask = topics == t
    fig.add_trace(
        go.Scattergl(
            x=umap_2d[mask, 0],
            y=umap_2d[mask, 1],
            mode="markers",
            marker=dict(size=4, color=color_map[t], opacity=0.5 if t == -1 else 0.7),
            name=topic_label(t, n_words=1),
            text=hover_texts[mask],
            hovertemplate="%{text}<extra></extra>",
        )
    )

fig.update_layout(
    title="Norwegian Usenet message embeddings (color=BERTopic topic)",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
    width=1100,
    height=750,
    legend=dict(font=dict(size=9)),
)
fig.show()